# Pelajaran 18: Mengamankan Agen AI dengan Bukti Kriptografi

## Buku Catatan Praktik

Buku catatan ini membahas empat latihan:

1. **Tandatangani bukti pertama Anda** untuk panggilan alat agen dan verifikasi.
2. **Ubah bukti** dan lihat verifikasi gagal.
3. **Buat rantai tiga bukti** dan konfirmasi integritas rantai.
4. **Bungkus panggilan alat Microsoft Agent Framework** sehingga setiap tindakan menghasilkan bukti.

Semua primitif kriptografi diimpor dari perpustakaan yang dikelola dengan baik (`pynacl` untuk Ed25519, `jcs` untuk RFC 8785 canonical JSON, `hashlib` dari pustaka standar Python untuk SHA-256). Logika bukti itu sendiri adalah Python biasa yang dapat Anda baca dan modifikasi.

Jalankan sel-sel sesuai urutan. Setiap bagian pendek dan mandiri.


## Setup

Pasang kedua dependensi. Keduanya memiliki lisensi permisif (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Utilitas Bantuan

Kedua pembantu ini menangani pengkodean base64url (tanpa padding) dan hashing SHA-256 dari objek arbitrer. Mereka menjaga agar bagian lain dari notebook tetap fokus pada logika tanda terima itu sendiri.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Bagian 1: Tanda tangani tanda terima pertama Anda

Bayangkan agen kami untuk **Contoso Travel** baru saja mencari penerbangan dari Sydney ke Los Angeles untuk seorang pelanggan. Kami ingin merekam panggilan alat ini sebagai tanda terima yang ditandatangani sehingga auditor di masa depan dapat memverifikasinya tanpa harus mempercayai kami.

### Langkah 1.1: Hasilkan kunci penandatanganan

Dalam produksi, kunci penandatanganan agen akan disimpan di modul keamanan perangkat keras (HSM), Azure Key Vault, atau penyimpanan terlindungi serupa. Untuk pelajaran ini kami menghasilkan kunci baru di memori.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Langkah 1.2: Bangun payload struk

Payload berisi semua yang ingin kita pastikan oleh struk: siapa yang bertindak, alat apa, dengan argumen apa, apa yang dikembalikan, di bawah kebijakan apa, dan kapan. Kami melakukan hash pada argumen dan hasilnya daripada memasukkannya secara langsung agar struk tidak membocorkan konten sensitif.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Langkah 1.3: Tanda tangani dan susun struk

Tiga langkah:

1. Kanonisasikan payload menggunakan JCS sehingga dua implementasi yang menghasilkan struk logis yang sama menghasilkan byte yang identik.
2. Hash byte kanonis dengan SHA-256.
3. Tanda tangani hash dengan kunci pribadi Ed25519.

Tanda tangan kemudian dipasang pada payload asli untuk menghasilkan struk akhir.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Langkah 1.4: Verifikasi tanda terima

Verifikasi membalikkan proses. Kami menghapus tanda tangan, menghitung ulang hash kanonik, dan memeriksa tanda tangan terhadap kunci publik dalam tanda terima.

Seorang auditor yang melakukan verifikasi ini tidak membutuhkan apa pun dari kami selain tanda terima itu sendiri. Tidak ada layanan yang harus dipanggil, tidak ada direktori kunci yang harus ditanyakan, tidak ada kepercayaan yang diperlukan.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Anda harus melihat `Receipt is valid: True`. Agen telah menghasilkan catatan audit yang ditandatangani secara kriptografis untuk pertama kalinya.


## Bagian 2: Memanipulasi tanda terima

Seluruh tujuan tanda terima adalah agar dapat terlihat jika telah dimanipulasi. Mari buktikan.

Kita akan mengubah tepat satu karakter pada tanda terima dan melihat verifikasi gagal.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Apa yang baru saja terjadi?

Ketika kami mengubah `policy_id`, byte kanonik berubah. Hash SHA-256 dari byte tersebut berubah. Tanda tangan (yang sebelumnya dibuat dari hash asli) tidak lagi cocok dengan hash baru. Verifikasi dengan benar mengembalikan `False`.

Tidak ada cara untuk memodifikasi bidang apa pun dari tanda terima dan masih dapat diverifikasi, kecuali penyerang memiliki kunci privat. Selama kunci privat disimpan dalam brankas kunci dan kunci publik dipublikasikan, manipulasi tidak mungkin disembunyikan.

Coba sendiri: ubah `tool_name` atau `agent_id` atau `timestamp` di sel di atas dan jalankan ulang. Setiap perubahan menghasilkan tanda terima yang tidak valid.


## Section 3: Rangkaian tanda terima bersama

Satu tanda terima melindungi satu tindakan. Sebagian besar agen melakukan banyak tindakan. Untuk membuat seluruh rangkaian tersebut tampak jika dirusak, kami menghubungkan setiap tanda terima ke tanda terima sebelumnya dengan menyertakan hash tanda terima sebelumnya dalam payload tanda terima baru.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Jika seseorang menghapus atau mengubah urutan tanda terima, rantainya akan putus tepat di titik itu. Verifikasi tanda terima selanjutnya gagal karena `previous_receipt_hash` tidak lagi cocok dengan hash sebenarnya dari pendahulunya.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Sekarang putuskan rantai dengan merusak struk tengah dan verifikasi ulang. Struk yang dirusak gagal pemeriksaan tanda tangannya, DAN struk berikutnya gagal pemeriksaan kaitan rantainya (karena `previous_receipt_hash` tidak lagi cocok dengan hash struk tengah yang dimodifikasi).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Struk 0 masih terverifikasi (tidak diubah dan tidak memiliki penerus yang harus diandalkan). Struk 1 gagal dalam pemeriksaan tanda tangannya karena kami mengubah `tool_args_hash`. Struk 2 gagal dalam pemeriksaan rantai tautan karena `previous_receipt_hash` dihitung terhadap struk 1 yang asli (yang kini sudah diubah).

Bahkan jika penyerang menandatangani ulang struk 1 yang sudah diubah (yang tidak bisa mereka lakukan tanpa kunci privat), ketidaksesuaian rantai tautan pada struk 2 tetap akan mengungkap manipulasi tersebut. Untuk menyembunyikan perubahan, penyerang harus menandatangani ulang setiap struk mulai dari titik perubahan, yang memerlukan kepemilikan kunci privat.


## Bagian 4: Membungkus panggilan alat agen dengan penandatanganan kuitansi

Dalam penerapan nyata, Anda tidak ingin setiap pembuat agen harus mengingat untuk memanggil `make_receipt`. Anda ingin penandatanganan kuitansi terjadi secara otomatis untuk setiap pemanggilan alat.

Berikut adalah pola paling sederhana: kelas pembungkus yang mengambil fungsi alat yang dapat dipanggil apa pun dan mengembalikan versi yang mengeluarkan kuitansi. Ini beradaptasi dengan kerangka agen apa pun, termasuk Kerangka Agen Microsoft (`agent_framework.azure`).

Jika Anda belum memiliki proyek Azure AI Foundry yang disiapkan, contoh lokal di bawah ini masih menunjukkan pola tersebut.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Integrasi dengan Microsoft Agent Framework

Pembungkus `ReceiptedTool` di atas bersifat framework-agnostik. Untuk menggunakannya di dalam agen yang dibangun dengan Microsoft Agent Framework, daftarkan fungsi yang dibungkus sebagai alat. Sebuah sketsa (Anda akan mengganti mock dengan pendaftaran alat Azure AI Foundry yang sebenarnya):

```python
# Pseudocode yang menunjukkan bentuk integrasi.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="Anda adalah agen Perjalanan Contoso ...",
#     tools=[receipted_lookup],   # alat yang dibungkus, bukan fungsi mentah
# )
# response = agent.run("Cari penerbangan dari Sydney ke Los Angeles pada bulan Juni.")
#
# # Setelah menjalankan, setiap panggilan alat yang dilakukan agen memiliki tanda terima yang ditandatangani:
# audit_chain = receipted_lookup.receipts
```

Framework agen tidak perlu mengetahui apa pun tentang tanda terima. Penandatanganan tanda terima dibungkus di sekitar alat, bukan dipasangkan ke dalam framework. Inilah cara Anda menambahkan asal usul ke kode agen yang ada tanpa menulis ulang agen tersebut.


## Rekap dan tantangan lanjutan

Anda telah:

- Menghasilkan pasangan kunci Ed25519.
- Membuat dan menandatangani tanda terima untuk panggilan alat agen.
- Memverifikasi tanda terima secara offline hanya menggunakan kunci publik.
- Memanipulasi tanda terima dan mengamati kegagalan verifikasi.
- Membuat rangkaian tanda terima yang dihubungkan dengan hash sebanyak tiga.
- Memanipulasi bagian tengah rantai dan mengamati kegagalan tanda tangan serta kegagalan tautan rantai.
- Membungkus fungsi alat dengan penandatanganan tanda terima otomatis.

**Tantangan lanjutan.** Perluas skema tanda terima dengan bidang `request_id` (UUID untuk pelacakan terdistribusi). Perbarui `make_receipt` untuk menyertakannya, dan pastikan tanda terima masih dapat diverifikasi dari ujung ke ujung. Kemudian ubah bidang tersebut setelah penandatanganan dan pastikan verifikasi gagal. Ini memaksa Anda untuk memahami bagaimana setiap byte dari pengkodean kanonis berkontribusi pada tanda tangan.

**Batas penting.** Tanda terima membuktikan tiga hal dan hanya tiga hal: atribusi (kunci ini menandatangani konten ini), integritas (konten tidak berubah sejak penandatanganan), dan urutan (tanda terima ini datang setelah tanda terima itu). Mereka TIDAK membuktikan bahwa tindakan agen benar, bahwa kebijakan yang disebutkan dalam `policy_id` benar-benar dievaluasi, atau bahwa agen mengikuti setiap aturan. Tanda terima adalah fondasi. Tata kelola adalah sistem yang Anda bangun di atasnya.

Baca kembali README pelajaran dengan batasan tersebut dalam pikiran. Kesalahan paling umum yang dilakukan tim dengan tanda terima adalah menganggap "kami memiliki tanda terima" berarti "kami diawasi." Tidak demikian. Tanda terima membuat perilaku agen dapat diaudit. Mereka tidak menjamin kebenarannya.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
